# Phase 4 and Phase 5 - Persistent backend, validation, and pulmonary segmentation

This notebook integrates the Phase 4 workflow (persistent storage, patient flow, metadata extraction, validation) with the Phase 5 prototype for pulmonary segmentation using MONAI.

Important: the segmentation step in this notebook is a technical prototype based on synthetic 3D data. It is not a clinical or diagnostic model.


## Phase 4 - Persistent storage and end-to-end workflow

The system stores patients and scans in a persistent SQLite database under the `data/` folder. This keeps the workflow stable across restarts and supports upload and metadata extraction.


In [ ]:
from pathlib import Path
import sqlite3

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
UPLOAD_DIR = DATA_DIR / 'uploads'
DB_PATH = DATA_DIR / 'clinical.db'

DATA_DIR.mkdir(exist_ok=True)
UPLOAD_DIR.mkdir(exist_ok=True)

conn = sqlite3.connect(DB_PATH)
conn.execute('''
CREATE TABLE IF NOT EXISTS patients (
    id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    age INTEGER,
    sex TEXT
);
''')
conn.execute('''
CREATE TABLE IF NOT EXISTS scans (
    id TEXT PRIMARY KEY,
    patient_id TEXT NOT NULL,
    file_name TEXT NOT NULL,
    file_path TEXT NOT NULL,
    file_type TEXT NOT NULL,
    dimensions TEXT NOT NULL,
    voxel_spacing TEXT NOT NULL,
    origin TEXT,
    orientation TEXT
);
''')
conn.commit()
conn.close()

print(f'Persistent database initialized at: {DB_PATH}')
print(f'Upload directory ready at: {UPLOAD_DIR}')


In [ ]:
import os
import numpy as np
import nibabel as nib
from pathlib import Path

def parse_nifti_metadata(file_path):
    img = nib.load(file_path)
    array = np.asarray(img.get_fdata(dtype=np.float32))
    affine = img.affine
    header = img.header
    return {
        'file_name': os.path.basename(file_path),
        'file_type': 'nifti',
        'dimensions': list(array.shape),
        'voxel_spacing': [float(x) for x in header.get_zooms()[:3]],
        'origin': [float(x) for x in affine[:3, 3]],
        'orientation': [float(x) for x in affine[:3, :3].reshape(-1)]
    }

shape = (64, 64, 32)
volume = np.zeros(shape, dtype=np.float32)
volume[18:46, 18:46, 8:24] = 1.0
trial_path = Path('data') / 'synthetic_volume.nii.gz'
trial_path.parent.mkdir(exist_ok=True)
nib.save(nib.Nifti1Image(volume, np.eye(4)), trial_path)

print(parse_nifti_metadata(str(trial_path)))


In [ ]:
from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from pathlib import Path
from uuid import uuid4

ROOT = Path.cwd()
UPLOAD_DIR = ROOT / 'data' / 'uploads'
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

app = FastAPI(title='MONAI Physio - Clinical Digital Twin Backend')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])
app.mount('/static/uploads', StaticFiles(directory=str(UPLOAD_DIR)), name='uploads')

@app.get('/api/health')
async def health_check():
    return {'status': 'healthy', 'services': {'api': 'active', 'database': 'sqlite_persistent'}}

@app.post('/api/patients')
async def create_patient(payload: dict):
    patient_id = payload.get('id') or str(uuid4())
    name = payload.get('name')
    if not name:
        raise HTTPException(status_code=400, detail='Patient name is required.')
    return {'message': 'Patient created', 'patient': {'id': patient_id, 'name': name}}

@app.post('/api/patients/{patient_id}/upload-scan')
async def upload_scan(patient_id: str, file: UploadFile = File(...)):
    if not file.filename:
        raise HTTPException(status_code=400, detail='No file provided')
    if not file.filename.lower().endswith(('.nii', '.nii.gz')):
        raise HTTPException(status_code=400, detail='Only NIfTI files are currently supported.')
    save_path = UPLOAD_DIR / file.filename
    with save_path.open('wb') as f:
        f.write(await file.read())
    return {'message': 'Scan uploaded successfully', 'patient_id': patient_id, 'file_path': str(save_path)}

print('FastAPI endpoints initialized successfully.')


In [ ]:
import sqlite3
import numpy as np
import nibabel as nib
from pathlib import Path

DB_PATH = Path('data/clinical.db')
volume = np.zeros((64, 64, 32), dtype=np.float32)
volume[18:46, 18:46, 8:24] = 1.0
nib.save(nib.Nifti1Image(volume, np.eye(4)), 'data/synthetic_volume.nii.gz')

conn = sqlite3.connect(DB_PATH)
conn.execute("INSERT OR REPLACE INTO patients (id, name, age, sex) VALUES (?, ?, ?, ?)", ('PAT-0001', 'Demo Patient', 57, 'F'))
conn.execute("INSERT OR REPLACE INTO scans (id, patient_id, file_name, file_path, file_type, dimensions, voxel_spacing, origin, orientation) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)", ('SCAN-0001', 'PAT-0001', 'synthetic_volume.nii.gz', 'data/synthetic_volume.nii.gz', 'nifti', str([64, 64, 32]), str([1.0, 1.0, 1.0]), str([0.0, 0.0, 0.0]), str([1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0])))
conn.commit()
patient = conn.execute('SELECT * FROM patients WHERE id = ?', ('PAT-0001',)).fetchone()
scan = conn.execute('SELECT * FROM scans WHERE patient_id = ?', ('PAT-0001',)).fetchone()
conn.close()

assert patient is not None
assert scan is not None
print('Patient record:', patient)
print('Scan record:', scan)
print('Phase 4 validation passed.')


## Phase 5 - Pulmonary segmentation with MONAI

This phase validates a MONAI-based 3D segmentation pipeline on synthetic lung-like volumes. It is a technical prototype only and must not be interpreted as a clinical diagnostic system.


In [ ]:
import sys
import subprocess
from pathlib import Path

for package in ['monai', 'torch', 'nibabel', 'numpy', 'matplotlib']:
    try:
        __import__(package)
        print(f'{package}: already installed')
    except Exception:
        print(f'Installing {package}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

import torch
import numpy as np
import monai
print('PyTorch:', torch.__version__)
print('MONAI:', monai.__version__)
print('CUDA available:', torch.cuda.is_available())


In [ ]:
from pathlib import Path
import numpy as np

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
MODELS_DIR = ROOT / 'models'
OUTPUTS_DIR = ROOT / 'outputs'
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

def generate_lung_mask(shape=(64, 64, 32), seed=0):
    rng = np.random.default_rng(seed)
    mask = np.zeros(shape, dtype=np.float32)
    x = np.arange(shape[0])[:, None, None]
    y = np.arange(shape[1])[None, :, None]
    z = np.arange(shape[2])[None, None, :]
    cx1, cy1, cz1 = shape[0] * 0.32, shape[1] * 0.50, shape[2] * 0.50
    cx2, cy2, cz2 = shape[0] * 0.68, shape[1] * 0.50, shape[2] * 0.50
    a1, b1, c1 = shape[0] * 0.20, shape[1] * 0.26, shape[2] * 0.42
    a2, b2, c2 = shape[0] * 0.20, shape[1] * 0.26, shape[2] * 0.42
    lung1 = ((x - cx1) / a1) ** 2 + ((y - cy1) / b1) ** 2 + ((z - cz1) / c1) ** 2 <= 1.0
    lung2 = ((x - cx2) / a2) ** 2 + ((y - cy2) / b2) ** 2 + ((z - cz2) / c2) ** 2 <= 1.0
    mask[lung1 | lung2] = 1.0
    noise = rng.normal(0.0, 0.05, size=shape)
    mask = np.clip(mask + noise, 0, 1).astype(np.float32)
    return mask

samples = []
for i in range(40):
    image = np.random.rand(* (64, 64, 32)).astype(np.float32) * 0.5
    mask = generate_lung_mask((64, 64, 32), seed=i)
    image = image + (mask * 1.5)
    image = np.clip(image, 0.0, 1.0)
    image_path = DATA_DIR / f'synthetic_lung_image_{i}.npy'
    mask_path = DATA_DIR / f'synthetic_lung_mask_{i}.npy'
    np.save(image_path, image)
    np.save(mask_path, mask)
    samples.append({'image': str(image_path), 'label': str(mask_path)})

print(f'Generated {len(samples)} synthetic samples in {DATA_DIR}')


In [ ]:
from monai.networks.nets import UNet
from monai.losses import DiceLoss
from monai.data import Dataset, DataLoader
from monai.transforms import Compose, EnsureType, AddChannel, ScaleIntensity
import numpy as np
import torch

class LungSegmentationDataset(Dataset):
    def __init__(self, items):
        self.items = items
        self.transforms = Compose([lambda x: x.astype(np.float32), AddChannel(), ScaleIntensity(), EnsureType()])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        image = np.load(item['image']).astype(np.float32)
        label = np.load(item['label']).astype(np.float32)
        image = self.transforms(image)[0]
        label = self.transforms(label)[0]
        return image, label

items = [
    {'image': str(p), 'label': str(q)}
    for p, q in zip(sorted(Path('data').glob('synthetic_lung_image_*.npy')), sorted(Path('data').glob('synthetic_lung_mask_*.npy')))
]
train_items = items[:32]
val_items = items[32:]
train_ds = LungSegmentationDataset(train_items)
val_ds = LungSegmentationDataset(val_items)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet(dimensions=3, in_channels=1, out_channels=1, channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2), kernel_size=3, up_kernel_size=3, dropout=0.1).to(device)
criterion = DiceLoss(to_onehot_y=False, sigmoid=True, squared_pred=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
print('Device:', device)
print(model)


In [ ]:
import torch
from pathlib import Path

epochs = 8
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if batch_idx % 2 == 0:
            print(f'Epoch {epoch + 1}/{epochs} | Batch {batch_idx} | Loss: {loss.item():.4f}')
    print(f'Epoch {epoch + 1}/{epochs} finished | Mean loss: {running_loss / max(1, len(train_loader)):.4f}')

model_path = Path('models') / 'lung_segmentation_unet.pt'
torch.save(model.state_dict(), model_path)
print(f'Model saved to {model_path}')


In [ ]:
import numpy as np
import torch
from pathlib import Path

model.eval()
with torch.no_grad():
    val_image, val_label = val_ds[0]
    val_input = val_image.unsqueeze(0).to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
    pred = model(val_input)
    prob = torch.sigmoid(pred)
    pred_mask = (prob > 0.5).float()
    agreement = (pred_mask.cpu().squeeze() == val_label).float().mean().item()
    print(f'Binary agreement on validation sample: {agreement:.4f}')
    np.save(Path('outputs') / 'synthetic_segmentation_prediction.npy', pred_mask.cpu().squeeze().numpy())
    np.save(Path('outputs') / 'synthetic_segmentation_reference.npy', val_label.numpy())
    print('Saved synthetic segmentation outputs under outputs/')


## Final synthesis

### Achievements
- Persistent SQLite database and upload pipeline for patient scans.
- Metadata extraction for NIfTI files.
- Validation of a complete synthetic patient-to-scan workflow.
- MONAI 3D segmentation prototype on synthetic lung-like volumes.

### Scientific limits
- This is not a clinically validated pulmonary segmentation model.
- The training data are synthetic and must be replaced by real CT volumes and verified model weights before production use.
- The digital twin remains a prototype workflow, not a diagnostic medical system.

### Phase 6 target
The next phase will connect the segmentation output to a 3D reconstruction pipeline and then to respiratory motion simulation, while preserving a strict separation between synthetic prototypes and real clinical data.
